In [1]:
import csv
import re
import time
from pathlib import Path
from urllib.parse import urljoin
from tqdm import tqdm
import requests
from bs4 import BeautifulSoup

In [3]:
BASE = "https://www.biostars.org"
TAG_INDEX_URL = BASE + "/t/?page={page}"
MAX_PAGES = 115
SLEEP_SEC = 0.4
OUT_CSV = Path("../02-datasource/biostars_all_tags.csv")
HEADERS = {
    "User-Agent": "BiostarsTagCollector/0.1 (+research use; contact: you@example.com)"
}

COUNT_RX = re.compile(r"[×x]\s*([\d,]+)")  # handles “× 1234” or "x 1,234"

def fetch(url: str) -> str:
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def parse_tag_page(html: str):
    """
    Yield tuples of (tag_name, count, href_abs) from a tag index page.
    """
    soup = BeautifulSoup(html, "html.parser")
    container = soup.select_one("div.tag-list")  # .ui.seven.column.tag-list.grid
    if not container:
        # Fallback: just scan all .item blocks
        items = soup.select("div.item")
    else:
        items = container.select("div.item")

    for item in items:
        a = item.select_one("a.ptag[href]")
        if not a:
            continue
        tag = a.get_text(strip=True)
        href_abs = urljoin(BASE, a["href"])
        text = item.get_text(" ", strip=True)
        m = COUNT_RX.search(text)
        count = int(m.group(1).replace(",", "")) if m else 0
        if tag:
            yield tag, count, href_abs

def main():
    tag_counts = {}  # tag -> count
    tag_href = {}    # tag -> absolute url

    for p in tqdm(range(1, MAX_PAGES + 1), desc="pages"):
        url = TAG_INDEX_URL.format(page=p)
        try:
            html = fetch(url)
        except Exception as e:
            print(f"[warn] failed to fetch page {p}: {e}")
            time.sleep(SLEEP_SEC)
            continue

        found = 0
        for tag, cnt, href in parse_tag_page(html):
            found += 1
            # If a tag appears multiple times, keep the max count (shouldn’t happen, but safe)
            prev = tag_counts.get(tag, 0)
            if cnt > prev:
                tag_counts[tag] = cnt
                tag_href[tag] = href

        # print(f"page {p}: {found} tags")
        time.sleep(SLEEP_SEC)

        # Optional early stop: if a page has zero tags, break (uncomment if desired)
        # if found == 0:
        #     break

    # Write CSV: category (tag name) and number of articles
    OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    with OUT_CSV.open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["category", "count", "url"])
        for tag in sorted(tag_counts):
            w.writerow([tag, tag_counts[tag], tag_href.get(tag, "")])

    print(f"wrote: {OUT_CSV}  (tags={len(tag_counts)})")

if __name__ == "__main__":
    main()


pages: 100%|██████████| 115/115 [03:12<00:00,  1.68s/it]

wrote: ../02-datasource/biostars_all_tags.csv  (tags=3168)
